<a href="https://colab.research.google.com/github/yoshiagent/vibration-colab/blob/main/notebooks/TwoDOF_Systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 開啟"/></a>

# 二自由度系統與動態吸振器 Two-DOF Systems and Vibration Absorber

對照網頁：[振動學｜二自由度系統](https://yoshiagent.github.io/vibration/MD.html)

| 原檔 | 內容 | 本筆記本 |
|---|---|---|
| `dof2free.m` | $M\ddot x+Kx=0$ 的模態解，六組初始條件（原程式用註解切換） | 第 1 節 |
| `Dof2mkForced.m` | 簡諧力作用下的振幅 $\hat x_1(\omega)$、$\hat x_2(\omega)$ | 第 2 節 |
| `Dof2mkAbsorber.m` | 加上吸振器（$m_2$、$k_2$）前後的主質量振幅 | 第 3 節 |

`eig(K,M)` 對應 `scipy.linalg.eigh(K, M)`（兩者都把特徵向量正規化成 $V^TMV=I$）。只用 Colab 內建的 numpy、scipy、plotly。

In [1]:
import numpy as np
from scipy.linalg import eigh
from scipy.integrate import solve_ivp
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def nz(v):
    """把 -0.0000 這類捨入誤差顯示成 0"""
    return 0.0 if abs(v) < 5e-9 else v


def show(fig, title, xlab="t (s)", ylab="x", h=420):
    fig.update_layout(title=title, xaxis_title=xlab, yaxis_title=ylab, height=h, template="plotly_white",
                      margin=dict(l=60, r=20, t=50, b=50))
    fig.show()

## 1. 自由振動與模態（`dof2free.m`）

```matlab
m1=1; m2=1; k1=4; k2=4; k3=4;
x0=[1;1]; dx0=[4*sqrt(3);0];          % case 6（case 1~5 在原程式裡是註解，改註解切換）
M=[m1 0;0 m2];  K=[k1+k2 -k2;-k2 k2+k3];
[V D]=eig(K,M);
w1=sqrt(D(1,1)); w2=sqrt(D(2,2)); v1=V(:,1); v2=V(:,2);
b13=inv(V)*x0;  b24=inv([w1*v1 w2*v2])*dx0;
xmod1=v1*(b1*cos(w1*t)+b2*sin(w1*t));  xmod2=v2*(b3*cos(w2*t)+b4*sin(w2*t));
x=xmod1+xmod2;
```

In [2]:
m1 = m2 = 1.0
k1 = k2 = k3 = 4.0
M = np.diag([m1, m2])
K = np.array([[k1 + k2, -k2], [-k2, k2 + k3]])
D, V = eigh(K, M)
w1, w2 = np.sqrt(D)
v1, v2 = V[:, 0], V[:, 1]
print(f"w1 = {w1:.4f} rad/s (= 2), w2 = {w2:.4f} rad/s (= 2 sqrt(3) = {2 * np.sqrt(3):.4f})")
print(f"v1 = {np.round(v1, 4)}, v2 = {np.round(v2, 4)}   (MATLAB-style normalization V'MV = I: proportional to [1, 1] and [1, -1])")
print(f"characteristic equation w^4 - 16 w^2 + 48 = 0 -> w^2 = {np.round(np.sort(np.roots([1, -16, 48])), 4)}")

cases = {1: ([1, 0], [0, 0]), 2: ([1, 1], [0, 0]), 3: ([1, -1], [0, 0]),
         4: ([1, 1], [4, 4]), 5: ([1, -1], [4, -4]), 6: ([1, 1], [4 * np.sqrt(3), 0])}
t = np.arange(0, 10 + 1e-9, 0.01)
results = {}
for n, (x0, dx0) in cases.items():
    x0, dx0 = np.array(x0, float), np.array(dx0, float)
    b1, b3 = np.linalg.solve(V, x0)
    b2, b4 = np.linalg.solve(np.column_stack([w1 * v1, w2 * v2]), dx0)
    xmod1 = np.outer(v1, b1 * np.cos(w1 * t) + b2 * np.sin(w1 * t))
    xmod2 = np.outer(v2, b3 * np.cos(w2 * t) + b4 * np.sin(w2 * t))
    x = xmod1 + xmod2
    sol = solve_ivp(lambda t, z: np.r_[z[2:], -np.linalg.solve(M, K @ z[:2])], (0, 10), np.r_[x0, dx0], t_eval=t, rtol=1e-11, atol=1e-12)
    # 換成網頁用的模態向量 [1, 1]、[1, -1]（未正規化）時的係數
    s1, s2 = v1[0], v2[0]
    print(f"case {n}: x0 = {x0}, dx0 = {np.round(dx0, 4)} -> (b1, b2, b3, b4) with modes [1,1],[1,-1] = "
          f"({nz(b1 * s1):.4f}, {nz(b2 * s1):.4f}, {nz(b3 * s2):.4f}, {nz(b4 * s2):.4f}); max |modal - solve_ivp| = {np.max(np.abs(x - sol.y[:2])):.1e}")
    results[n] = (x, xmod1, xmod2)

w1 = 2.0000 rad/s (= 2), w2 = 3.4641 rad/s (= 2 sqrt(3) = 3.4641)
v1 = [-0.7071 -0.7071], v2 = [-0.7071  0.7071]   (MATLAB-style normalization V'MV = I: proportional to [1, 1] and [1, -1])
characteristic equation w^4 - 16 w^2 + 48 = 0 -> w^2 = [ 4. 12.]
case 1: x0 = [1. 0.], dx0 = [0. 0.] -> (b1, b2, b3, b4) with modes [1,1],[1,-1] = (0.5000, 0.0000, 0.5000, 0.0000); max |modal - solve_ivp| = 2.7e-11
case 2: x0 = [1. 1.], dx0 = [0. 0.] -> (b1, b2, b3, b4) with modes [1,1],[1,-1] = (1.0000, 0.0000, 0.0000, 0.0000); max |modal - solve_ivp| = 2.6e-11


case 3: x0 = [ 1. -1.], dx0 = [0. 0.] -> (b1, b2, b3, b4) with modes [1,1],[1,-1] = (0.0000, 0.0000, 1.0000, 0.0000); max |modal - solve_ivp| = 4.7e-11
case 4: x0 = [1. 1.], dx0 = [4. 4.] -> (b1, b2, b3, b4) with modes [1,1],[1,-1] = (1.0000, 2.0000, 0.0000, 0.0000); max |modal - solve_ivp| = 5.7e-11
case 5: x0 = [ 1. -1.], dx0 = [ 4. -4.] -> (b1, b2, b3, b4) with modes [1,1],[1,-1] = (0.0000, 0.0000, 1.0000, 1.1547); max |modal - solve_ivp| = 6.4e-11


case 6: x0 = [1. 1.], dx0 = [6.9282 0.    ] -> (b1, b2, b3, b4) with modes [1,1],[1,-1] = (1.0000, 1.7321, 0.0000, 1.0000); max |modal - solve_ivp| = 7.1e-11


網頁講義 Case 5 原本寫 $b_4=2$，由 $2\sqrt3\,b_4=4$ 應為 $b_4=2/\sqrt3\approx1.1547$（上面 case 5 的輸出），網頁已加編註更正。
只有初始位移與初始速度都和某一個模態成比例時（case 2、3、4、5），響應才只含那一個頻率；case 1、6 兩個模態都有。
下面畫 case 6（原程式最後啟用的那一組）的五張圖：

In [3]:
x, xmod1, xmod2 = results[6]
fig = make_subplots(rows=3, cols=2, subplot_titles=("x1 (red), x2 (blue)", "x1 = x1 mode1 + x1 mode2", "x2 = x2 mode1 + x2 mode2",
                                                    "mode 1: x1 (red), x2 (blue)", "mode 2: x1 (red), x2 (blue)", ""),
                    vertical_spacing=0.09)
fig.add_trace(go.Scatter(x=t, y=x[0], line=dict(color="red"), name="x1"), 1, 1)
fig.add_trace(go.Scatter(x=t, y=x[1], line=dict(color="blue"), name="x2"), 1, 1)
fig.add_trace(go.Scatter(x=t, y=x[0], line=dict(color="red"), showlegend=False), 1, 2)
fig.add_trace(go.Scatter(x=t, y=xmod1[0], line=dict(color="green"), name="mode components"), 1, 2)
fig.add_trace(go.Scatter(x=t, y=xmod2[0], line=dict(color="green", dash="dot"), showlegend=False), 1, 2)
fig.add_trace(go.Scatter(x=t, y=x[1], line=dict(color="blue"), showlegend=False), 2, 1)
fig.add_trace(go.Scatter(x=t, y=xmod1[1], line=dict(color="green"), showlegend=False), 2, 1)
fig.add_trace(go.Scatter(x=t, y=xmod2[1], line=dict(color="green", dash="dot"), showlegend=False), 2, 1)
fig.add_trace(go.Scatter(x=t, y=xmod1[0], line=dict(color="red"), showlegend=False), 2, 2)
fig.add_trace(go.Scatter(x=t, y=xmod1[1], line=dict(color="blue", dash="dot"), showlegend=False), 2, 2)
fig.add_trace(go.Scatter(x=t, y=xmod2[0], line=dict(color="red"), showlegend=False), 3, 1)
fig.add_trace(go.Scatter(x=t, y=xmod2[1], line=dict(color="blue"), showlegend=False), 3, 1)
show(fig, "dof2free.m, case 6: x0 = [1; 1], dx0 = [4 sqrt(3); 0]", xlab=None, ylab=None, h=820)

六個案例一起看（下拉選單切換）：

In [4]:
fig = go.Figure()
for n in cases:
    fig.add_trace(go.Scatter(x=t, y=results[n][0][0], name="x1", line=dict(color="red"), visible=(n == 1)))
    fig.add_trace(go.Scatter(x=t, y=results[n][0][1], name="x2", line=dict(color="blue"), visible=(n == 1)))
fig.update_layout(updatemenus=[dict(x=1.0, y=1.15, buttons=[
    dict(label=f"case {n}", method="update", args=[{"visible": [j // 2 == n - 1 for j in range(12)]}]) for n in cases])])
show(fig, "dof2free.m: response for the six initial conditions")

## 2. 強迫響應（`Dof2mkForced.m`）

```matlab
m1=1; m2=1; k1=1; k2=2; k3=2; f1=2; f2=-1; w=0:0.01:4;
detw=(k1+k2-m1*w.*w).*(k2+k3-m2*w.*w)-k2*k2;
x1=((k2+k3-m2*w.*w)*f1+k2*f2)./detw;
x2=((k1+k2-m1*w.*w)*f2+k2*f1)./detw;
```
$\Delta(\omega)=0$ 的兩個根就是兩個自然頻率（共振）；分子為 0 的頻率，該質量的振幅為 0（反共振）。

In [5]:
m1, m2, k1, k2, k3, f1, f2 = 1.0, 1.0, 1.0, 2.0, 2.0, 2.0, -1.0
w = np.arange(0, 4 + 1e-9, 0.01)
detw = (k1 + k2 - m1 * w * w) * (k2 + k3 - m2 * w * w) - k2 * k2
x1 = ((k2 + k3 - m2 * w * w) * f1 + k2 * f2) / detw
x2 = ((k1 + k2 - m1 * w * w) * f2 + k2 * f1) / detw
Kf, Mf = np.array([[k1 + k2, -k2], [-k2, k2 + k3]]), np.diag([m1, m2])
wn = np.sqrt(eigh(Kf, Mf, eigvals_only=True))
print(f"natural frequencies (Delta = 0): {np.round(wn, 4)} rad/s")
anti = np.sqrt((k2 + k3) / m2 + k2 * f2 / (m2 * f1))
print(f"x1 = 0 (anti-resonance) at w = sqrt((k2+k3)/m2 + k2 f2/(m2 f1)) = {anti:.4f} rad/s")
err = max(np.max(np.abs(np.linalg.solve(Kf - wk**2 * Mf, [f1, f2]) - [x1[i], x2[i]])) for i, wk in enumerate(w) if i % 7 == 0 and abs(detw[i]) > 1e-3)
print(f"closed form vs solve((K - w^2 M), f): max difference = {err:.1e}")
for xx, name in ((x1, "x1"), (x2, "x2")):
    fig = go.Figure(go.Scatter(x=w, y=np.where(np.abs(xx) > 50, np.nan, xx), name=name))
    for wv in wn:
        fig.add_vline(x=wv, line_dash="dot", line_color="gray")
    fig.update_yaxes(range=[-3, 3])
    show(fig, f"Dof2mkForced.m: amplitude {name} vs w (axis [0 4 -3 3])", xlab="w (rad/s)", ylab=name)

natural frequencies (Delta = 0): [1.1994 2.3583] rad/s
x1 = 0 (anti-resonance) at w = sqrt((k2+k3)/m2 + k2 f2/(m2 f1)) = 1.7321 rad/s
closed form vs solve((K - w^2 M), f): max difference = 7.1e-15


## 3. 動態吸振器（`Dof2mkAbsorber.m`）

```matlab
m1=100; k1=900; f1=1;  kmratio=k1/m1;
m2=10; %m2=100;        k2=m2*kmratio;           % 吸振器調在 sqrt(k2/m2) = sqrt(k1/m1) = 3 rad/s
w=1:0.01:5;
x10=f1./(k1-m1*w.*w)*100;                       % 原系統（cm）
detw=(k1+k2-m1*w.*w).*(k2-m2*w.*w)-k2*k2;
wn1=roots([m1*m2 0 -(k1+k2)*m2-m1*k2 0 k1*k2])  % 加吸振器後的兩個自然頻率
x1=((k2-m2*w.*w)*f1)./detw*100;  x2=(k2*f1)./detw*100;
```
網頁講義的吸振器 $\hat x_1$ 分子是 $(k_2-\omega^2m_2)\hat f_1$（講義圖上誤寫成 $k_2+k_3$，已編註）；這和程式一致。

In [6]:
m1, k1, f1 = 100.0, 900.0, 1.0
w = np.arange(1, 5 + 1e-9, 0.01)
x10 = f1 / (k1 - m1 * w * w) * 100
print(f"wn0 = roots([-m1 0 k1]) = {np.round(np.sort(np.roots([-m1, 0, k1])), 4)}")
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, subplot_titles=("Original system X1 (cm)", "Absorber system X1 (cm)", "Absorber system X2 (cm)"))
fig.add_trace(go.Scatter(x=w, y=np.where(np.abs(x10) > 10, np.nan, x10), name="original x1", line=dict(color="gray")), 1, 1)
for m2, color in ((10.0, "blue"), (100.0, "orange")):
    k2 = m2 * k1 / m1
    detw = (k1 + k2 - m1 * w * w) * (k2 - m2 * w * w) - k2 * k2
    x1 = (k2 - m2 * w * w) * f1 / detw * 100
    x2 = k2 * f1 / detw * 100
    wn1 = np.roots([m1 * m2, 0, -(k1 + k2) * m2 - m1 * k2, 0, k1 * k2])
    pos = np.sort(wn1[wn1 > 0].real)[::-1]
    print(f"m2 = {m2:g}, k2 = {k2:g}: wn1 = {np.round(pos, 2)} rad/s;  at w = 3: x1 = {nz((k2 - m2 * 9) * f1 / ((k1 + k2 - m1 * 9) * (k2 - m2 * 9) - k2 * k2) * 100):.4f} cm, "
          f"x2 = {k2 * f1 / ((k1 + k2 - m1 * 9) * (k2 - m2 * 9) - k2 * k2) * 100:.4f} cm (= -f1/k2 = {-f1 / k2 * 100:.4f} cm)")
    fig.add_trace(go.Scatter(x=w, y=np.where(np.abs(x1) > 10, np.nan, x1), name=f"x1, m2 = {m2:g}", line=dict(color=color)), 2, 1)
    fig.add_trace(go.Scatter(x=w, y=np.where(np.abs(x2) > 10, np.nan, x2), name=f"x2, m2 = {m2:g}", line=dict(color=color)), 3, 1)
for r in (1, 2, 3):
    fig.update_yaxes(range=[-3, 3], row=r, col=1)
fig.update_xaxes(title_text="w (rad/s)", row=3, col=1)
show(fig, "Dof2mkAbsorber.m: absorber tuned to 3 rad/s (m2 = 10 and the commented-out m2 = 100)", xlab=None, ylab=None, h=820)

wn0 = roots([-m1 0 k1]) = [-3.  3.]
m2 = 10, k2 = 90: wn1 = [3.51 2.56] rad/s;  at w = 3: x1 = 0.0000 cm, x2 = -1.1111 cm (= -f1/k2 = -1.1111 cm)
m2 = 100, k2 = 900: wn1 = [4.85 1.85] rad/s;  at w = 3: x1 = 0.0000 cm, x2 = -0.1111 cm (= -f1/k2 = -0.1111 cm)


吸振器把 3 rad/s 的共振變成「主質量不動」，代價是在兩側各出現一個新的共振頻率；
$m_2$ 越大（$k_2$ 同比例變大），兩個新自然頻率離 3 rad/s 越遠、可用的頻帶越寬，吸振器的振幅 $|\hat x_2|=\hat f/k_2$ 也越小。
網頁講義的第二組數值 $k_2=9$、$m_2=1$（同樣調在 3 rad/s）：

In [7]:
for m2, k2 in ((10.0, 90.0), (1.0, 9.0)):
    wn1 = np.sqrt(eigh(np.array([[k1 + k2, -k2], [-k2, k2]]), np.diag([m1, m2]), eigvals_only=True))[::-1]
    print(f"k2 = {k2:g}, m2 = {m2:g}: natural frequencies = {wn1[0]:.2f} and {wn1[1]:.2f} rad/s")

k2 = 90, m2 = 10: natural frequencies = 3.51 and 2.56 rad/s
k2 = 9, m2 = 1: natural frequencies = 3.15 and 2.85 rad/s
